# Hello Chatbot with Groq API

**Week 1 · Class 2** — Python refresher + Groq API

## Learning objectives

- Refresh **Python basics**: print, variables, if, loops, functions
- Explain what Groq provides and why we use it
- Make a chat completion API call in Python
- Understand **message roles** (system, user, assistant)
- Build a simple **Hello Chatbot** REPL
- Complete the **Gradio** chat UI assignment (Section 16 — you write the code)

Get a free API key at [console.groq.com](https://console.groq.com) before Section 7.

> Run cells **top to bottom**. Code matches the Class 2 slides.


## Section 1 — Print

Use `print()` to show text in the console. Strings go in quotes.

In [ ]:
print("Hello, chatbot bootcamp!")
print("You can print numbers too:", 42)
topic = "chatbots"
print(f"Today we learn about {topic}.")


## Section 2 — Variables

Store values in **names** so you can reuse them.

In [ ]:
name = "Alex"
lesson = 2
print(f"Student: {name}, Class: {lesson}")
name = "Sam"  # you can change a variable
print("Updated name:", name)


## Section 3 — If statements

Run code only when a condition is true. You will use this in the chat loop (`quit`).

In [ ]:
user_input = ""
if not user_input:
    print("Please type something.")
elif user_input.lower() == "quit":
    print("Goodbye!")
else:
    print("You said:", user_input)


## Section 4 — Loops

Repeat code with `for` or `while`.

In [ ]:
words = ["Hello", "Chatbot", "Bootcamp"]
for word in words:
    print(word)

count = 0
while count < 3:
    print("Count:", count)
    count += 1


## Section 5 — Functions

A **function** groups steps you can call again. Later you will write `chat()` and `ask()`.

In [ ]:
def greet(name: str) -> str:
    return f"Hello, {name}!"

print(greet("bootcamper"))
print(greet("Groq"))


## Section 6 — Setup

Install the official Groq Python client.

In [ ]:
!pip install -q groq



## Section 7 — API key

Never commit API keys to Git. In Colab you can:

1. **This cell** — enter your key when prompted (uses `getpass`), or
2. **Colab Secrets** — store `GROQ_API_KEY` in the key icon sidebar.

If you see **401 Unauthorized** later, re-run this cell with a valid key.


In [ ]:
import os
import getpass

from google.colab import userdata
api_key=userdata.get('GROQ_API_KEY')


## Section 8 — Minimal API call

One **user** message → one assistant reply.

> Verify the model name on [Groq docs](https://console.groq.com/docs/models).


In [ ]:
from groq import Groq

client = Groq(api_key=api_key)
MODEL = "llama-3.3-70b-versatile"

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is a chatbot?"}],
)

print(response.choices[0].message.content)


## Section 9 — Message roles at a glance

| Role | Who writes it | Purpose |
|------|---------------|--------|
| **system** | Developer (hidden from end user in most UIs) | Instructions: tone, role, format, rules |
| **user** | Human | The person's question or command |
| **assistant** | The model | Previous replies — needed for **multi-turn** memory |

Every API call sends a **list** of messages. Order matters.


## Section 9c — `chat(messages)` helper

One function for all role patterns. Pass the full `messages` list.


In [ ]:
def chat(messages: list) -> str:
    """Send a messages list to Groq and return the assistant reply text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
    )
    return response.choices[0].message.content


## Section 9a — Same question, different system prompt

The **user** message stays the same. Only the **system** instruction changes.

> The model did not get smarter — the **instruction** changed.


In [ ]:
QUESTION = "Explain what a chatbot is in two sentences."

print("=== User only (no system) ===")
print(chat([{"role": "user", "content": QUESTION}]))

print("\n=== Tutor system ===")
print(chat([
    {"role": "system", "content": "You are a patient programming tutor. Use simple steps."},
    {"role": "user", "content": QUESTION},
]))

print("\n=== Comedian system ===")
print(chat([
    {"role": "system", "content": "You are a witty comedian. Add light humor. Keep it short."},
    {"role": "user", "content": QUESTION},
]))


## Section 9b — Multi-turn: assistant = memory

Include prior **user** and **assistant** messages so the model remembers context.


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "My favorite color is blue."},
]

first_reply = chat(messages)
print("Assistant:", first_reply)

messages.append({"role": "assistant", "content": first_reply})
messages.append({"role": "user", "content": "What is my favorite color?"})

print("\nFollow-up:", chat(messages))


## Section 10 — Response anatomy

Extract the reply text and optional token usage.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in five words."}],
)

text = response.choices[0].message.content
usage = response.usage

print("Reply:", text)
print("Prompt tokens:", usage.prompt_tokens)
print("Completion tokens:", usage.completion_tokens)


### Optional — measure latency (Section 11)

Rough timing for one call (RPM, TPM, 429 errors in class).


In [ ]:
import time

start = time.time()
_ = chat([{"role": "user", "content": "Say hi in five words."}])
print(f"Round-trip: {time.time() - start:.2f}s")


## Section 12 — `ask()` wrapper

Convenience helper for single user questions (no system prompt).


In [ ]:
def ask(question: str) -> str:
    return chat([{"role": "user", "content": question}])


## Section 13 — Single test


In [ ]:
print(ask("What is a chatbot?"))



## Section 14 — Hello Chatbot loop

Type questions below. Enter **quit** or **exit** to stop.


In [ ]:
print("Hello Chatbot! Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye!")
        break
    print("Bot:", ask(user_input), "\n")


## Section 15 — Error handling


In [ ]:
def ask_safe(question: str) -> str:
    try:
        return chat([{"role": "user", "content": question}])
    except Exception as e:
        return f"Sorry, something went wrong: {e}"


In [ ]:
print(ask_safe("Hello!"))



## Section 16 — Assignment: Gradio chat UI (you implement)

**Gradio** turns your Python function into a web chat UI. This section is an **assignment** — there is no solution code in the notebook. You write it yourself after class.

### What to build

1. Run `!pip install gradio` in a **new code cell** you create below.
2. Define a `SYSTEM_PROMPT` string (personality / rules for the bot).
3. Write a `respond(user_message, history)` function that:
   - Starts `messages` with `{"role": "system", "content": SYSTEM_PROMPT}`
   - For each `[user_turn, bot_turn]` in `history`, append user and assistant messages
   - Appends the new `user_message` as a user turn
   - Returns `chat(messages)` (use the `chat()` function from Section 9c)
4. Create `gr.ChatInterface(fn=respond, title="Hello Chatbot (Groq)")` and call `.launch()`.

### Colab tips

- If the UI does not appear: try `.launch(debug=True)` or `.launch(share=True)` for a public link.
- API key cells (Section 7) must run first.

### Deliverables

- Working Gradio app with a **system prompt**
- **3+ example chats** tested
- **Screenshot** or `share=True` URL for demo
- **Optional:** try two different `SYSTEM_PROMPT` values and compare replies

Docs: [gradio.app/docs](https://www.gradio.app/docs)

### Pseudocode (structure only — not runnable)

```
SYSTEM_PROMPT = "..."
def respond(user_message, history):
    messages = [system message]
    for each past turn in history:
        append user message
        append assistant message
    append new user message
    return chat(messages)
gr.ChatInterface(fn=respond, ...).launch()
```


## Section 17 — Homework checklist

- [ ] Complete notebook Sections 1–15 in class (Python refresher through `ask_safe`)
- [ ] **Assignment:** Section 16 — build your own **Gradio** app (no starter code in repo)
- [ ] Test **3+ example chats** in your Gradio UI
- [ ] Save a **screenshot** or `share=True` public URL for your demo
- [ ] **Optional:** two different `SYSTEM_PROMPT` values and compare replies
- [ ] Bring one error you hit (if any) to **Class 3** Q&A

**Next class (Class 3):** personalized chatbot with **system prompts** in the **CLI** (teacher bot, joke bot, travel guide).
